# Tipos relacionados a UUID

É comum usar UUIDs como chave dos dados (muitas vezes em bancos de dados).

O Python tem várias funções de UUID; a mais comum é a UUID4.

In [1]:
from uuid import uuid4

In [2]:
print(f"uuid4(): {uuid4()!r}")

uuid4(): UUID('402dd965-d0f8-474d-8fdd-082732241f41')


Podemos facilmente declarar campos desse tipo:

In [3]:
from pydantic import BaseModel, Field, UUID4, ValidationError

In [4]:
class Person(BaseModel):
    id: UUID4

In [5]:
p = Person(id=uuid4())
print(f"p: {p!r}")

p: Person(id=UUID('78faf7eb-1fcb-48e1-b747-ada1b14df92a'))


E ao serializar:

In [6]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id': UUID('78faf7eb-1fcb-48e1-b747-ada1b14df92a')}


In [7]:
print(f"p.model_dump_json(): {p.model_dump_json()!r}")

p.model_dump_json(): '{"id":"78faf7eb-1fcb-48e1-b747-ada1b14df92a"}'


E se quisermos um valor default para `id`?

Podemos simplesmente fazer isto?

In [8]:
class Person(BaseModel):
    id: UUID4 = uuid4()

In [9]:
p1 = Person()
print(f"p1: {p1!r}")

p1: Person(id=UUID('af2ee2f8-4793-412e-aeec-493545b76293'))


In [10]:
p2 = Person()
print(f"p2: {p2!r}")

p2: Person(id=UUID('af2ee2f8-4793-412e-aeec-493545b76293'))


Viu o problema? Recebemos o **mesmo** UUID4 para instâncias diferentes.

In [11]:
print(f"p1.id == p2.id: {p1.id == p2.id!r}")

p1.id == p2.id: True


Vamos voltar a isso com mais detalhe, mas vejamos rapidamente aqui.

Outra coisa de que você pode não gostar é usar `id` como nome de campo (e, mesmo que não se importe, os linters vão reclamar!).

Então devemos usar um alias — para isso precisamos de um `Field`, e podemos ver se usar `default` no `Field` resolve o problema:

In [12]:
class Person(BaseModel):
    id_: UUID4 = Field(alias="id", default=uuid4())

In [13]:
p1 = Person()
print(f"p1: {p1!r}")

p1: Person(id_=UUID('a1072605-760b-49f3-9601-7fe6f99f36fb'))


In [14]:
p2 = Person()
print(f"p2: {p2!r}")

p2: Person(id_=UUID('a1072605-760b-49f3-9601-7fe6f99f36fb'))


Usar `default` no `Field` não resolveu.

O problema é o mesmo de usar uma chamada de função para gerar o default de um parâmetro de função.

O default é criado uma vez, quando a função é criada. O mesmo vale para `Field`: é criado uma vez, quando o modelo é criado — por isso
vemos o mesmo UUID.

Precisamos que o default seja gerado **a cada** criação de instância.

Fazemos isso com o argumento `default_factory` do `Field`.

Ele recebe um *callable* que o Pydantic chama para cada instância que precisa de um default.

Testando:

In [15]:
class Person(BaseModel):
    id_: UUID4 = Field(alias="id", default_factory=uuid4)

Repare que passamos o *callable* em `default_factory` — **não** chamamos a função. O Pydantic a chama quando precisa de um novo default.

In [16]:
p1 = Person()
print(f"p1: {p1!r}")

p1: Person(id_=UUID('36f6ce1c-1dc4-4135-8ece-c68dc2148432'))


In [17]:
p2 = Person()
print(f"p2: {p2!r}")

p2: Person(id_=UUID('40855c17-abbb-4293-bd1e-ad4b6ba306f7'))


In [18]:
print(f"p1.id_ != p2.id_: {p1.id_ != p2.id_!r}")

p1.id_ != p2.id_: True


Sucesso!

O Pydantic também suporta outros tipos de UUID, como UUID1, UUID3 e UUID5 (e, nas versões recentes, UUID6/7/8).

> 💡 **Complemento:** `UUID4` valida a **versão**. Um UUID válido de outra versão é rejeitado:

In [19]:
from uuid import uuid1

try:
    Person(id=uuid1())
except ValidationError as ex:
    print(f'ex.errors()[0]["msg"]: {ex.errors()[0]["msg"]}')

ex.errors()[0]["msg"]: UUID version 4 expected
